# 🚗 Predicting Electric Vehicle Purchases — CatBoost Multi-Seed Ensemble

**Competition:** [Playground Series — Season 6, Episode 9](https://www.kaggle.com/competitions/playground-series-s6e9)

**Metric:** ROC AUC

---

### 📋 Pipeline Overview

| Step | Description |
|------|-------------|
| 1 | **Exploratory Data Analysis** — Target distribution, feature correlations, missing values |
| 2 | **Feature Engineering** — Type-aware preprocessing, categorical encoding |
| 3 | **Model Training** — CatBoost with 3-seed × 5-fold = 15 model ensemble |
| 4 | **Submission** — Averaged probability predictions |

> 💡 **Key Idea:** Multi-seed ensembling with early stopping reduces variance and overfitting. CatBoost handles categoricals natively — no label encoding needed.

---

In [ ]:
# ── Cell 1: Imports ──────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
from catboost import CatBoostClassifier, Pool
from sklearn.metrics import roc_auc_score, roc_curve
from sklearn.model_selection import StratifiedKFold
import warnings, pathlib, json, logging

warnings.filterwarnings('ignore')
logging.basicConfig(level=logging.INFO, format='%(levelname)s %(message)s')
LOGGER = logging.getLogger(__name__)

# ── Aesthetics ────────────────────────────────────────────────────────────────
plt.rcParams.update({
    'figure.facecolor': '#0e1117',
    'axes.facecolor': '#0e1117',
    'axes.edgecolor': '#333333',
    'axes.labelcolor': '#e0e0e0',
    'text.color': '#e0e0e0',
    'xtick.color': '#aaaaaa',
    'ytick.color': '#aaaaaa',
    'grid.color': '#222222',
    'figure.figsize': (14, 5),
    'font.size': 11,
    'font.family': 'sans-serif',
})
PALETTE = ['#00d2ff', '#ff6b6b', '#51cf66', '#ffd43b', '#cc5de8', '#ff922b']

print('✅ Imports loaded')

In [ ]:
# ── Cell 2: Load Data ────────────────────────────────────────────────────────────
ON_KAGGLE = pathlib.Path('/kaggle').exists()
DATA_DIR = pathlib.Path('/kaggle/input/playground-series-s6e9') if ON_KAGGLE else pathlib.Path('data')

def find_file(root, name):
    matches = sorted(root.glob(f'**/{name}'), key=lambda p: (len(p.parts), str(p)))
    if not matches:
        raise FileNotFoundError(f'{name} not found under {root}')
    return matches[0]

train = pd.read_csv(find_file(DATA_DIR, 'train.csv'))
test  = pd.read_csv(find_file(DATA_DIR, 'test.csv'))
sample_sub = pd.read_csv(find_file(DATA_DIR, 'sample_submission.csv'))

TARGET = 'Will_Buy_EV'
ID_COL = 'id'

print(f'Train shape: {train.shape}')
print(f'Test shape:  {test.shape}')
print(f'Target distribution:\n{train[TARGET].value_counts(normalize=True).to_string()}')
train.head()

## 📊 Section 1: Exploratory Data Analysis

Understanding the data before modeling is critical for feature engineering decisions.

In [ ]:
# ── Cell 3: Target Distribution ──────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Plot 1: Target balance
counts = train[TARGET].value_counts()
bars = axes[0].bar(counts.index.astype(str), counts.values, color=[PALETTE[0], PALETTE[1]], 
                   edgecolor='white', linewidth=0.5, width=0.5)
for bar, val in zip(bars, counts.values):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 100, 
                 f'{val:,}', ha='center', fontsize=12, fontweight='bold', color='white')
axes[0].set_title('Target Distribution', fontsize=14, fontweight='bold')
axes[0].set_xlabel(TARGET)
axes[0].set_ylabel('Count')

# Plot 2: Missing values
missing = train.isnull().sum().sort_values(ascending=False)
missing = missing[missing > 0]
if len(missing) > 0:
    axes[1].barh(missing.index[:15], missing.values[:15], color=PALETTE[3], edgecolor='white', linewidth=0.3)
    axes[1].set_title('Missing Values (Top 15)', fontsize=14, fontweight='bold')
else:
    axes[1].text(0.5, 0.5, '✅ No missing values!', transform=axes[1].transAxes, 
                ha='center', va='center', fontsize=16, color=PALETTE[2], fontweight='bold')
    axes[1].set_title('Missing Values', fontsize=14, fontweight='bold')

# Plot 3: Feature types
dtype_counts = train.dtypes.value_counts()
axes[2].pie(dtype_counts.values, labels=dtype_counts.index.astype(str), 
           autopct='%1.0f%%', colors=PALETTE[:len(dtype_counts)],
           textprops={'color': 'white', 'fontsize': 11})
axes[2].set_title('Feature Types', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
# ── Cell 4: Numeric Feature Distributions ────────────────────────────────────────
numeric_cols = train.select_dtypes(include=[np.number]).columns.drop([ID_COL, TARGET], errors='ignore')

n_cols = 4
n_rows = (len(numeric_cols) + n_cols - 1) // n_cols
fig, axes = plt.subplots(n_rows, n_cols, figsize=(20, 4 * n_rows))
axes = axes.flatten()

for idx, col in enumerate(numeric_cols):
    ax = axes[idx]
    for label, color in zip(train[TARGET].unique(), [PALETTE[0], PALETTE[1]]):
        subset = train[train[TARGET] == label][col].dropna()
        ax.hist(subset, bins=40, alpha=0.6, color=color, label=f'{TARGET}={label}', density=True)
    ax.set_title(col, fontsize=11, fontweight='bold')
    ax.legend(fontsize=8)

for idx in range(len(numeric_cols), len(axes)):
    axes[idx].set_visible(False)

plt.suptitle('Numeric Feature Distributions by Target', fontsize=16, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# ── Cell 5: Correlation Heatmap ──────────────────────────────────────────────────
corr_cols = list(numeric_cols) + [TARGET]
corr = train[corr_cols].corr()

fig, ax = plt.subplots(figsize=(12, 10))
mask = np.triu(np.ones_like(corr, dtype=bool), k=1)
cmap = sns.diverging_palette(220, 20, as_cmap=True)
sns.heatmap(corr, mask=mask, cmap=cmap, center=0, vmin=-1, vmax=1,
            square=True, linewidths=0.5, annot=True, fmt='.2f', annot_kws={'size': 8},
            cbar_kws={'shrink': 0.8, 'label': 'Pearson Correlation'}, ax=ax)
ax.set_title('Feature Correlation Matrix', fontsize=16, fontweight='bold', pad=15)
plt.tight_layout()
plt.show()

# Top correlations with target
target_corr = corr[TARGET].drop(TARGET).abs().sort_values(ascending=False)
print(f'\n🔑 Top 10 Features Correlated with {TARGET}:')
for feat, val in target_corr.head(10).items():
    print(f'   {feat:30s} | r = {val:.4f}')

## 🛠️ Section 2: Feature Engineering & Preprocessing

In [ ]:
# ── Cell 6: Feature Preparation ──────────────────────────────────────────────────
feature_cols = [c for c in train.columns if c not in {TARGET, ID_COL}]
print(f'Features: {len(feature_cols)}')

def prepare_features(df, cols):
    """Type-aware feature preparation for CatBoost."""
    result = df[cols].copy()
    for col in result.columns:
        if pd.api.types.is_bool_dtype(result[col]):
            result[col] = result[col].astype('int8')
        elif pd.api.types.is_object_dtype(result[col]) or pd.api.types.is_string_dtype(result[col]):
            result[col] = result[col].astype('string').fillna('__MISSING__').astype(str)
        elif isinstance(result[col].dtype, pd.CategoricalDtype):
            result[col] = result[col].astype(str).replace('nan', '__MISSING__')
    return result

X_train = prepare_features(train, feature_cols)
X_test  = prepare_features(test, feature_cols)
y = train[TARGET]

cat_features = [i for i, c in enumerate(X_train.columns) if X_train[c].dtype == object]
print(f'Categorical features ({len(cat_features)}): {[X_train.columns[i] for i in cat_features]}')
print(f'Numeric features: {len(feature_cols) - len(cat_features)}')
print(f'\nX_train shape: {X_train.shape}')
print(f'X_test shape:  {X_test.shape}')

## 🚀 Section 3: Multi-Seed CatBoost Ensemble Training

We train **3 seeds × 5 folds = 15 models** and average predictions for a robust ensemble.

> 🎯 **Why multi-seed?** Different random seeds explore different decision boundaries. Averaging across seeds reduces variance and guards against overfitting to any particular fold split.

In [ ]:
# ── Cell 7: Training Loop ────────────────────────────────────────────────────────
SEEDS = [42, 2026, 31415]
N_FOLDS = 5

oof = np.zeros(len(train), dtype=float)
test_preds = np.zeros(len(test), dtype=float)
fold_scores = []
importances = pd.DataFrame(index=feature_cols)

for seed_idx, seed in enumerate(SEEDS):
    skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=seed)
    for fold, (train_idx, val_idx) in enumerate(skf.split(X_train, y), 1):
        model = CatBoostClassifier(
            loss_function='Logloss', eval_metric='AUC',
            iterations=4000, learning_rate=0.03,
            depth=8, l2_leaf_reg=8.0,
            random_strength=0.5, bagging_temperature=0.5,
            border_count=254, random_seed=seed + fold,
            thread_count=-1, allow_writing_files=False, verbose=False,
        )
        model.fit(
            X_train.iloc[train_idx], y.iloc[train_idx],
            cat_features=cat_features,
            eval_set=(X_train.iloc[val_idx], y.iloc[val_idx]),
            use_best_model=True, early_stopping_rounds=200, verbose=False,
        )
        
        classes = np.asarray(model.classes_)
        pos_idx = list(classes).index(1) if 1 in classes else 0
        
        val_pred = model.predict_proba(X_train.iloc[val_idx])[:, pos_idx]
        tst_pred = model.predict_proba(X_test)[:, pos_idx]
        
        oof[val_idx] += val_pred / len(SEEDS)
        test_preds += tst_pred / (len(SEEDS) * N_FOLDS)
        
        score = roc_auc_score(y.iloc[val_idx], val_pred)
        fold_scores.append(score)
        
        imp_key = f's{seed_idx}_f{fold}'
        importances[imp_key] = model.get_feature_importance()
        
        print(f'  Seed {seed} | Fold {fold}/{N_FOLDS} | Best iter: {model.get_best_iteration():4d} | AUC: {score:.6f}')

oof_score = roc_auc_score(y, oof)
print(f'\n{"="*60}')
print(f'🏆 OOF ROC AUC:        {oof_score:.6f}')
print(f'📊 Mean Fold ROC AUC:  {np.mean(fold_scores):.6f} ± {np.std(fold_scores):.6f}')
print(f'{"="*60}')

In [ ]:
# ── Cell 8: Training Results Visualization ───────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(20, 6))

# Plot 1: Fold AUC scores
colors = [PALETTE[i // N_FOLDS] for i in range(len(fold_scores))]
bars = axes[0].bar(range(len(fold_scores)), fold_scores, color=colors, 
                   edgecolor='white', linewidth=0.3)
axes[0].axhline(y=oof_score, color=PALETTE[1], linestyle='--', linewidth=2, label=f'OOF: {oof_score:.5f}')
axes[0].set_xlabel('Model Index (seed × fold)')
axes[0].set_ylabel('ROC AUC')
axes[0].set_title('Per-Fold ROC AUC Scores', fontsize=14, fontweight='bold')
axes[0].legend(fontsize=11)
axes[0].set_ylim(min(fold_scores) - 0.005, max(fold_scores) + 0.005)

# Plot 2: ROC Curve
fpr, tpr, _ = roc_curve(y, oof)
axes[1].plot(fpr, tpr, color=PALETTE[0], linewidth=2.5, label=f'ROC AUC = {oof_score:.5f}')
axes[1].plot([0, 1], [0, 1], '--', color='gray', linewidth=1)
axes[1].fill_between(fpr, tpr, alpha=0.15, color=PALETTE[0])
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate')
axes[1].set_title('ROC Curve (OOF)', fontsize=14, fontweight='bold')
axes[1].legend(fontsize=12, loc='lower right')

# Plot 3: Feature Importance (Top 15)
mean_imp = importances.mean(axis=1).sort_values(ascending=True)
top_imp = mean_imp.tail(15)
axes[2].barh(top_imp.index, top_imp.values, color=PALETTE[2], edgecolor='white', linewidth=0.3)
axes[2].set_title('Top 15 Feature Importances', fontsize=14, fontweight='bold')
axes[2].set_xlabel('Mean Importance')

plt.tight_layout()
plt.show()

In [ ]:
# ── Cell 9: Prediction Distribution ──────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# OOF predictions by class
for label, color in zip([0, 1], [PALETTE[0], PALETTE[1]]):
    mask = y == label
    axes[0].hist(oof[mask], bins=50, alpha=0.6, color=color, label=f'{TARGET}={label}', density=True)
axes[0].set_title('OOF Prediction Distribution', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Predicted Probability')
axes[0].legend()

# Test predictions
axes[1].hist(test_preds, bins=50, alpha=0.7, color=PALETTE[4], density=True)
axes[1].axvline(x=0.5, color=PALETTE[1], linestyle='--', linewidth=2, label='Decision boundary')
axes[1].set_title('Test Prediction Distribution', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Predicted Probability')
axes[1].legend()

plt.tight_layout()
plt.show()

## 📤 Section 4: Generate Submission

In [ ]:
# ── Cell 10: Create Submission CSV ───────────────────────────────────────────────
submission = sample_sub.copy()
submission[TARGET] = test_preds

# Validation checks
assert len(submission) == len(test), 'Row count mismatch!'
assert not submission[ID_COL].duplicated().any(), 'Duplicate IDs!'
assert not submission[TARGET].isna().any(), 'Missing predictions!'
assert submission[TARGET].between(0, 1).all(), 'Predictions out of [0,1] range!'

OUTPUT = pathlib.Path('/kaggle/working/submission.csv') if ON_KAGGLE else pathlib.Path('submission.csv')
OUTPUT.parent.mkdir(parents=True, exist_ok=True)
submission.to_csv(OUTPUT, index=False)

print(f'✅ Submission saved to {OUTPUT}')
print(f'   Rows: {len(submission):,}')
print(f'   Columns: {list(submission.columns)}')
print(f'   Prediction range: [{submission[TARGET].min():.6f}, {submission[TARGET].max():.6f}]')
print(f'   Prediction mean:  {submission[TARGET].mean():.6f}')
print(f'\n🏆 OOF ROC AUC: {oof_score:.6f}')

submission.head(10)

---

## 📌 Summary

| Metric | Value |
|--------|-------|
| Model | CatBoost (Logloss) |
| Ensemble | 3 seeds × 5 folds = 15 models |
| OOF ROC AUC | See above |
| Key features | See importance chart |

### 🔮 Potential Improvements
- Feature interaction engineering
- LightGBM / XGBoost stacking
- Target encoding for high-cardinality categoricals
- Pseudo-labeling with confident test predictions

---

**If you found this notebook helpful, please upvote! 👍**